# ViralScope AI - Training Notebook

This notebook trains the ViralScope multimodal model using thumbnail images and video titles.

**Tasks covered:**
- T4.4: Create PyTorch Dataset and DataLoader
- T4.5: Implement Training Loop with Metrics
- T4.6: Unfreeze Backbones for Fine-Tuning (optional)

**Runtime:** ~30-60 minutes with GPU

In [ ]:
#@title 1. Setup & Installation
!pip install kaggle pandas pillow requests tqdm transformers torch torchvision scikit-learn -q

In [ ]:
#@title 2. Clone Repository & Setup Paths

import os
import sys

# Clone the repository if not already done
if not os.path.exists('/content/ViralScope-AI'):
    !git clone https://github.com/yassine-cmd/ViralScope-AI.git /content/ViralScope-AI

# Add to path
sys.path.insert(0, '/content/ViralScope-AI')
os.chdir('/content/ViralScope-AI')

print(f"Working directory: {os.getcwd()}")
!ls -la

In [ ]:
#@title 3. Imports & Configuration

import os
import random
import yaml
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchvision import transforms
from transformers import AutoTokenizer
from PIL import Image

from sklearn.metrics import roc_auc_score, average_precision_score

# Configuration
with open('config.yaml', 'r') as f:
    CONFIG = yaml.safe_load(f)

print(f"Device: {CONFIG['project']['device']}")
print(f"Batch size: {CONFIG['training']['batch_size']}")
print(f"Epochs: {CONFIG['training']['epochs']}")

In [ ]:
#@title 4. Reproducibility

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['project']['seed'])
print("Random seed set to:", CONFIG['project']['seed'])

In [ ]:
#@title 5. Dataset Class

class ViralScopeDataset(Dataset):
    def __init__(self, split='train', transform=None, tokenizer=None):
        self.split = split
        self.transform = transform
        self.tokenizer = tokenizer
        
        self.data_dir = CONFIG['data']['processed_dir']
        self.thumbnail_dir = os.path.join(self.data_dir, 'thumbnails')
        
        self.df = self._load_split()
    
    def _load_split(self):
        df = pd.read_csv(os.path.join(self.data_dir, 'labeled_dataset.csv'))
        
        split_file = os.path.join(CONFIG['data']['splits_dir'], f'{self.split}_indices.pt')
        if os.path.exists(split_file):
            indices = torch.load(split_file).numpy()
            df = df.iloc[indices].reset_index(drop=True)
        else:
            print(f"Warning: {split_file} not found. Using all data.")
        
        return df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        image_path = os.path.join(self.thumbnail_dir, f"{row['video_id']}.jpg")
        try:
            image = Image.open(image_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (224, 224), color=(128, 128, 128))
        
        if self.transform:
            image = self.transform(image)
        
        # Tokenize title
        title = str(row.get('title', ''))
        if self.tokenizer:
            encoding = self.tokenizer(
                title,
                truncation=True,
                max_length=CONFIG['model']['nlp']['max_seq_length'],
                padding='max_length',
                return_tensors='pt'
            )
            input_ids = encoding['input_ids'].squeeze()
            attention_mask = encoding['attention_mask'].squeeze()
        else:
            input_ids = torch.zeros(CONFIG['model']['nlp']['max_seq_length'], dtype=torch.long)
            attention_mask = torch.zeros(CONFIG['model']['nlp']['max_seq_length'], dtype=torch.long)
        
        label = torch.tensor(row['is_viral'], dtype=torch.float32)
        
        return {
            'image': image,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': label
        }

print("ViralScopeDataset class defined")

In [ ]:
#@title 6. Load Models

from models.multimodal import ViralScopeModel
from models.losses import FocalLoss

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = ViralScopeModel(CONFIG).to(device)
print(f"Model loaded: ViralScopeModel")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
#@title 7. Setup DataLoaders

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model']['nlp']['checkpoint'])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = ViralScopeDataset(split='train', transform=train_transform, tokenizer=tokenizer)
val_dataset = ViralScopeDataset(split='val', transform=train_transform, tokenizer=tokenizer)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created")

In [ ]:
#@title 8. Setup Optimizer, Loss, Scheduler

# Separate backbone and head parameters for differential learning rates
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if 'fusion' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = AdamW([
    {'params': backbone_params, 'lr': CONFIG['model']['lr_backbone_cv']},
    {'params': head_params, 'lr': CONFIG['model']['lr_head']}
], weight_decay=CONFIG['model']['weight_decay'])

criterion = FocalLoss(
    alpha=None,
    gamma=CONFIG['training']['focal_loss_gamma']
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=CONFIG['training']['scheduler_factor'],
    patience=CONFIG['training']['scheduler_patience'],
    verbose=True
)

print("Optimizer: AdamW with differential learning rates")
print(f"  Backbone LR: {CONFIG['model']['lr_backbone_cv']}")
print(f"  Head LR: {CONFIG['model']['lr_head']}")

In [ ]:
#@title 9. Metrics Functions

def compute_metrics(preds, targets, threshold=0.5):
    preds_binary = (preds >= threshold).float()
    
    tp = ((preds_binary == 1) & (targets == 1)).sum().item()
    tn = ((preds_binary == 0) & (targets == 0)).sum().item()
    fp = ((preds_binary == 1) & (targets == 0)).sum().item()
    fn = ((preds_binary == 0) & (targets == 1)).sum().item()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-10)
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    
    try:
        auc_roc = roc_auc_score(targets.numpy(), preds.numpy())
        pr_auc = average_precision_score(targets.numpy(), preds.numpy())
    except:
        auc_roc = 0.0
        pr_auc = 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'pr_auc': pr_auc
    }

In [ ]:
#@title 10. Training Functions

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    for batch in dataloader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(images, input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        probs = torch.sigmoid(logits).detach()
        all_preds.extend(probs.cpu())
        all_targets.extend(labels.cpu())
    
    all_preds = torch.stack(all_preds)
    all_targets = torch.stack(all_targets)
    metrics = compute_metrics(all_preds, all_targets)
    metrics['loss'] = total_loss / len(dataloader)
    
    return metrics


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(images, input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu())
            all_targets.extend(labels.cpu())
    
    all_preds = torch.stack(all_preds)
    all_targets = torch.stack(all_targets)
    metrics = compute_metrics(all_preds, all_targets)
    metrics['loss'] = total_loss / len(dataloader)
    
    return metrics

In [ ]:
#@title 11. Training Loop

from google.colab import drive
import time

# Mount Google Drive for saving checkpoints
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/ViralScope/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCHS = CONFIG['training']['epochs']
EARLY_STOPPING_PATIENCE = CONFIG['training']['early_stopping_patience']

best_metric = 0.0
best_metrics = {}
early_stopping_counter = 0
training_history = []

print(f"\n{'='*80}")
print(f"Starting training for {EPOCHS} epochs")
print(f"{'='*80}\n")

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    # Train
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_metrics['pr_auc'])
    
    epoch_time = time.time() - epoch_start
    
    # Log
    log = {
        'epoch': epoch + 1,
        'train_loss': train_metrics['loss'],
        'train_acc': train_metrics['accuracy'],
        'train_f1': train_metrics['f1'],
        'train_pr_auc': train_metrics['pr_auc'],
        'val_loss': val_metrics['loss'],
        'val_acc': val_metrics['accuracy'],
        'val_f1': val_metrics['f1'],
        'val_pr_auc': val_metrics['pr_auc'],
        'time': epoch_time
    }
    training_history.append(log)
    
    print(f"Epoch {epoch+1:>3}/{EPOCHS} ({epoch_time:.1f}s) | "
          f"Train Loss: {train_metrics['loss']:.4f} | "
          f"Val Loss: {val_metrics['loss']:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"Val PR-AUC: {val_metrics['pr_auc']:.4f}")
    
    # Save best model
    if val_metrics['pr_auc'] > best_metric:
        best_metric = val_metrics['pr_auc']
        best_metrics = val_metrics.copy()
        early_stopping_counter = 0
        
        checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_metric': best_metric,
            'config': CONFIG
        }, checkpoint_path)
        print(f"  -> Saved best model (PR-AUC: {best_metric:.4f})")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs")
            break

print(f"\n{'='*80}")
print(f"Training complete! Best PR-AUC: {best_metric:.4f}")
print(f"{'='*80}")

In [ ]:
#@title 12. Save Training History

# Save training history to CSV
history_df = pd.DataFrame(training_history)
history_path = os.path.join(CHECKPOINT_DIR, 'training_history.csv')
history_df.to_csv(history_path, index=False)
print(f"Training history saved to: {history_path}")

# Display final metrics
print(f"\n{'='*50}")
print("Final Best Metrics:")
print(f"{'='*50}")
for key, value in best_metrics.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
#@title 13. Optional: Fine-Tuning (Unfreeze Backbones)
# Uncomment to fine-tune the backbones after training the head

# print("\nUnfreezing backbones for fine-tuning...")
# for param in model.cv_extractor.parameters():
#     param.requires_grad = True
# for param in model.nlp_extractor.parameters():
#     param.requires_grad = True

# # Lower learning rates for fine-tuning
# optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
# scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# # Continue training for a few more epochs
# for epoch in range(5):
#     # ... (same training loop as above)
print("Fine-tuning code ready - uncomment to use")

---

## Next Steps

After training:
1. Run `scripts/04_evaluate.py` or create an evaluation notebook
2. Run XAI notebooks for Grad-CAM and Integrated Gradients
3. Deploy using `app/gradio_app.py`